# Rekap Anomali Pertanggal

Script ini menggabungkan data anomali (usaha & keluarga) terbaru dengan rekap hasil hari-hari sebelumnya, lalu menandai status penyelesaian (resolved) per tanggal dalam bentuk kolom checklist.

**Alur:**
1. Ambil & gabungkan data anomali terbaru (usaha + keluarga)
2. Pisahkan anomali yang resolved vs belum resolved
3. Muat rekap terakhir (kalau ada), gabungkan dengan anomali yang belum resolved (outer join) -> baris baru & lama tetap ada
4. Tandai baris yang statusnya resolved hari ini
5. Hitung ringkasan perubahan dibanding rekap sebelumnya
6. Simpan hasil ke file Excel baru (sheet Rekap + Ringkasan)


## Import & Konfigurasi

In [1]:
from datetime import datetime
from pathlib import Path
import re
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

In [2]:
KEY_COLS = [
    "assignment_id",
    "kode_kabupaten",
    "level_6_code",
    "nama_tercantum",
    "source_type",
    "anomali_no",
]
EXTRA_COLS = ["extra_columns"]
SELECT_COLS = KEY_COLS + ["is_resolved"] + EXTRA_COLS

FOLDER_REKAP = Path("../rekap_anomali_pertanggal")
TANDA_CENTANG = "\u2713"
TANDA_KONDISI_LAPANGAN = "Kondisi Lapangan"
FORMAT_TANGGAL = "%d-%m-%Y"
FORMAT_TANGGAL_JAM = "%d-%m-%Y_%H-%M-%S"
MAX_LEBAR_KOLOM = 40

## Ambil Data

In [3]:
folder = Path("../scrap_anomali_usaha")
files = list(folder.glob("anomali_usaha_sumsel_*.xlsx"))
if not files:
    raise FileNotFoundError("Tidak ada file Excel di folder scrap_anomali_usaha")
def extract_timestamp(f):
    # sesuaikan regex ini dengan format timestamp di nama file kamu
    match = re.search(r"(\d{8}_\d{6})", f.stem)  # contoh: 20250706_123045
    return match.group(1) if match else ""
file_terbaru = max(files, key=extract_timestamp)
print("Membaca file:", file_terbaru)
df = pd.read_excel(file_terbaru, dtype=str)
print(len(df))

Membaca file: ..\scrap_anomali_usaha\anomali_usaha_sumsel_20260715_093840.xlsx
27313


In [5]:
POLA_TIMESTAMP = re.compile(r"(\d{8}_\d{6})")
FORMAT_TIMESTAMP = "%Y%m%d_%H%M%S"
def ambil_timestamp_dari_nama(path: Path) -> datetime:
    """Ekstrak timestamp dari nama file, contoh: anomali_keluarga_sumsel_20260630_115129.xlsx"""
    match = POLA_TIMESTAMP.search(path.stem)
    if not match:
        raise ValueError(f"Tidak menemukan timestamp pada nama file: {path.name}")
    return datetime.strptime(match.group(1), FORMAT_TIMESTAMP)


folder = Path("../scrap_anomali_keluarga")
files = list(folder.glob("anomali_keluarga_sumsel_*.xlsx"))
if not files:
    raise FileNotFoundError("Tidak ada file Excel di folder scrap_anomali_keluarga")
# Ambil file terbaru berdasarkan waktu modifikasi
file_terbaru = max(files, key=ambil_timestamp_dari_nama)
print("Membaca file:", file_terbaru)
df2 = pd.read_excel(file_terbaru, dtype=str)
print(len(df2))

Membaca file: ..\scrap_anomali_keluarga\anomali_keluarga_sumsel_20260715_095422.xlsx
128470


In [6]:
anomali_usaha = df[
    [
        "assignment_id",
        "kode_kabupaten",
        "level_6_code",
        "nama_tercantum",
        "source_type",
        "anomali_no",
        "is_resolved",
        "extra_columns",
    ]
]
anomali_keluarga = df2[
    [
        "assignment_id",
        "kode_kabupaten",
        "level_6_code",
        "nama_tercantum",
        "source_type",
        "anomali_no",
        "is_resolved",
        "extra_columns",
    ]
]

## 1. Persiapan Data Anomali Terbaru

In [7]:
def siapkan_raw_anomali(df_usaha: pd.DataFrame, df_keluarga: pd.DataFrame) -> pd.DataFrame:
    """Gabungkan data anomali usaha & keluarga, ambil kolom relevan saja."""
    anomali_usaha = df_usaha[SELECT_COLS]
    anomali_keluarga = df_keluarga[SELECT_COLS]
    return pd.concat([anomali_usaha, anomali_keluarga], ignore_index=True)


def pisahkan_status_resolved(raw_anomali: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Pisahkan anomali menjadi (is_resolved, not_resolved).

    Catatan: kolom is_resolved berisi string 'True'/'False', bukan boolean.
    """
    is_resolved = raw_anomali[raw_anomali["is_resolved"] == "True"]
    not_resolved = raw_anomali[raw_anomali["is_resolved"] == "False"]
    return is_resolved, not_resolved

## 2. Muat Rekap Sebelumnya

In [8]:
def muat_rekap_terakhir(folder_rekap: Path) -> pd.DataFrame | None:
    """Cari & muat file rekap terakhir berdasarkan waktu modifikasi.

    Return None kalau belum ada file rekap sama sekali.
    """
    existing_files = list(folder_rekap.glob("rekap_anomali_*.xlsx"))
    if not existing_files:
        print("Belum ada file rekap sebelumnya, akan membuat baru.")
        return None

    file_terakhir = max(existing_files, key=lambda f: f.stat().st_mtime)
    print("File rekap terakhir ditemukan:", file_terakhir)
    df_lama = pd.read_excel(file_terakhir, sheet_name="Rekap", dtype=str)
    # mask_blank = df_lama["30-06-2026"].fillna("").str.strip() == ""
    # df_lama = df_lama[mask_blank]
    return df_lama

## 3. Gabungkan Rekap Lama dengan Anomali Baru (Belum Resolved)

In [9]:
def gabungkan_dengan_rekap_lama(
    df_lama: pd.DataFrame | None,
    not_resolved: pd.DataFrame,
    tanggal_hari_ini: str,
) -> pd.DataFrame:
    """Outer join rekap lama dengan anomali yang belum resolved hari ini.

    Baris yang belum resolved diberi tanda centang pada kolom tanggal
    hari ini. Baris lama yang tidak muncul lagi akan tetap ada
    (kolom tanggal hari ini menjadi kosong/NaN untuk baris tsb).
    """
    raw_anomali_baru = not_resolved.drop(columns=["is_resolved"]).copy()
    raw_anomali_baru[tanggal_hari_ini] = TANDA_CENTANG

    if df_lama is None:
        print(f"Belum ada data sebelumnya, semua {len(raw_anomali_baru)} baris dianggap baru.")
        return raw_anomali_baru
    rekap = pd.merge(
        df_lama,
        raw_anomali_baru,
        on=KEY_COLS,
        how="outer",
        suffixes=("", "_baru"),
        indicator=True,
    )
    for col in EXTRA_COLS:
        col_baru = f"{col}_baru"
        if col_baru in rekap.columns:
            rekap[col] = rekap[col_baru].combine_first(rekap[col])
            rekap = rekap.drop(columns=[col_baru])
    return rekap.drop(columns=["_merge"])


def tandai_resolved(
    rekap: pd.DataFrame,
    is_resolved: pd.DataFrame,
    tanggal_hari_ini: str,
) -> pd.DataFrame:
    """Tandai baris yang key-nya cocok dengan anomali is_resolved.

    Hanya baris yang match yang diubah; baris lain (mis. sudah berisi
    tanda centang dari gabungkan_dengan_rekap_lama) tidak disentuh.
    """
    rekap = pd.merge(
        rekap,
        is_resolved[KEY_COLS],
        on=KEY_COLS,
        how="left",
        indicator="_merge_resolved",
    )
    rekap.loc[rekap["_merge_resolved"] == "both", tanggal_hari_ini] = TANDA_KONDISI_LAPANGAN
    return rekap.drop(columns=["_merge_resolved"])

## 4. Ringkasan Perubahan

In [10]:
def _is_kolom_tanggal(col: str) -> bool:
    try:
        datetime.strptime(col, FORMAT_TANGGAL)
        return True
    except (ValueError, TypeError):
        return False


def cari_tanggal_sebelumnya(rekap: pd.DataFrame, tanggal_hari_ini: str) -> str | None:
    """Cari kolom tanggal terakhir sebelum tanggal_hari_ini (untuk pembanding)."""
    kolom_tanggal = [c for c in rekap.columns if _is_kolom_tanggal(c)]
    kolom_tanggal_sorted = sorted(kolom_tanggal, key=lambda c: datetime.strptime(c, FORMAT_TANGGAL))
    kolom_sebelum_hari_ini = [c for c in kolom_tanggal_sorted if c != tanggal_hari_ini]

    print("Kolom tanggal terdeteksi:", kolom_tanggal_sorted)
    return kolom_sebelum_hari_ini[-1] if kolom_sebelum_hari_ini else None


def buat_ringkasan(
    rekap: pd.DataFrame,
    df_lama: pd.DataFrame | None,
    raw_anomali_baru_count: int,
    tanggal_hari_ini: str,
    tanggal_sebelumnya: str | None,
) -> pd.DataFrame:
    """Susun tabel ringkasan: jumlah data lama/baru & perubahan status centang."""
    jumlah_data_sebelumnya = len(df_lama) if df_lama is not None else 0

    if tanggal_sebelumnya is not None:
        kolom_now = rekap[tanggal_hari_ini].fillna("")
        kolom_prev = rekap[tanggal_sebelumnya].fillna("")

        was_centang = kolom_prev == TANDA_CENTANG
        now_centang = kolom_now == TANDA_CENTANG

        jumlah_berubah = (was_centang & ~now_centang).sum()
        jumlah_belum_berubah = (was_centang & now_centang).sum()
    else:
        jumlah_berubah = None
        jumlah_belum_berubah = None
        print("Belum ada kolom tanggal sebelumnya untuk dibandingkan (rekap pertama kali).")

    return pd.DataFrame(
        {
            "Keterangan": [
                "Data sebelumnya",
                "Data baru masuk",
                f"Perubahan (centang -> blank/{TANDA_KONDISI_LAPANGAN}), "
                f"{tanggal_sebelumnya} -> {tanggal_hari_ini}",
                f"Belum berubah (tetap centang), {tanggal_sebelumnya} -> {tanggal_hari_ini}",
                "Total setelah digabung",
            ],
            "Jumlah": [
                jumlah_data_sebelumnya,
                raw_anomali_baru_count,
                jumlah_berubah,
                jumlah_belum_berubah,
                len(rekap),
            ],
        }
    )

## 5. Simpan ke Excel

In [11]:
def _is_kolom_tanggal(col: str) -> bool:
    try:
        datetime.strptime(col, FORMAT_TANGGAL)
        return True
    except (ValueError, TypeError):
        return False

LINK_BASE_URL = "https://fasih-sm.bps.go.id/app/assignment/fd68e454-ba45-4b85-8205-f3bf777ded24/"  # ganti dengan URL asli kamu
LINK_COL = "hyperlink"


def tambah_kolom_link(rekap: pd.DataFrame) -> pd.DataFrame:
    """Tambah kolom link = LINK_BASE_URL + assignment_id."""
    rekap = rekap.copy()
    rekap[LINK_COL] = LINK_BASE_URL + rekap["assignment_id"].astype(str)
    return rekap

def urutkan_kolom_rekap(rekap: pd.DataFrame) -> pd.DataFrame:
    """Susun ulang urutan kolom: key -> extra_columns -> tanggal (kronologis)."""
    kolom_tanggal = [c for c in rekap.columns if _is_kolom_tanggal(c)]
    kolom_tanggal_sorted = sorted(
        kolom_tanggal, key=lambda c: datetime.strptime(c, FORMAT_TANGGAL)
    )

    kolom_urut = KEY_COLS + EXTRA_COLS + [LINK_COL] + kolom_tanggal_sorted
    # jaga-jaga kalau ada kolom tak terduga (mis. dari merge yang belum ke-drop), taruh di akhir
    kolom_sisa = [c for c in rekap.columns if c not in kolom_urut]
    return rekap[kolom_urut + kolom_sisa]


def _jadikan_hyperlink(
    path_file: Path, sheet_name: str = "Rekap", kolom_link: str = LINK_COL
) -> None:
    """Ubah teks URL di kolom link menjadi hyperlink asli yang bisa diklik."""
    wb = load_workbook(path_file)
    ws = wb[sheet_name]

    # cari posisi kolom berdasarkan header di baris pertama
    header = [cell.value for cell in ws[1]]
    if kolom_link not in header:
        wb.close()
        return
    col_idx = header.index(kolom_link) + 1  # openpyxl 1-indexed

    for row in range(2, ws.max_row + 1):  # skip header
        cell = ws.cell(row=row, column=col_idx)
        if cell.value:
            cell.hyperlink = cell.value
            cell.font = Font(color="0563C1", underline="single")

    wb.save(path_file)


def simpan_ke_excel(
    rekap: pd.DataFrame, ringkasan: pd.DataFrame, path_file: Path
) -> None:
    with pd.ExcelWriter(path_file, engine="openpyxl") as writer:
        rekap.to_excel(writer, sheet_name="Rekap", index=False)
        ringkasan.to_excel(writer, sheet_name="Ringkasan", index=False)

    _jadikan_hyperlink(path_file)  # <-- tambahkan ini
    _atur_lebar_kolom(path_file)


def _atur_lebar_kolom(path_file: Path, max_lebar: int = MAX_LEBAR_KOLOM) -> None:
    """Sesuaikan lebar kolom tiap sheet otomatis berdasarkan isi terpanjang."""
    wb = load_workbook(path_file)
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        for col_cells in ws.columns:
            max_len = max(
                (len(str(cell.value)) if cell.value is not None else 0) for cell in col_cells
            )
            col_letter = col_cells[0].column_letter
            ws.column_dimensions[col_letter].width = min(max_len + 2, max_lebar)
    wb.save(path_file)

## 6. Main — Jalankan Proses

In [12]:

FOLDER_REKAP.mkdir(parents=True, exist_ok=True)

tanggal_hari_ini = datetime.now().strftime(FORMAT_TANGGAL)
tanggal_jam = datetime.now().strftime(FORMAT_TANGGAL_JAM)

# 1. Siapkan data anomali terbaru
raw_anomali = siapkan_raw_anomali(df, df2)
is_resolved, not_resolved = pisahkan_status_resolved(raw_anomali)
print(f"Anomali resolved: {len(is_resolved)} | belum resolved: {len(not_resolved)}")

# 2. Muat rekap sebelumnya (kalau ada)
df_lama = muat_rekap_terakhir(FOLDER_REKAP)

# 3. Gabungkan & tandai status
rekap = gabungkan_dengan_rekap_lama(df_lama, not_resolved, tanggal_hari_ini)
rekap = tandai_resolved(rekap, is_resolved, tanggal_hari_ini)

# 4. Ringkasan perubahan
tanggal_sebelumnya = cari_tanggal_sebelumnya(rekap, tanggal_hari_ini)
ringkasan = buat_ringkasan(
    rekap,
    df_lama,
    raw_anomali_baru_count=len(not_resolved),
    tanggal_hari_ini=tanggal_hari_ini,
    tanggal_sebelumnya=tanggal_sebelumnya,
)
print(ringkasan.to_string(index=False))
rekap = tambah_kolom_link(rekap)  # <-- tambahkan ini
rekap = urutkan_kolom_rekap(rekap)
# (opsional) rapikan urutan & isi kosong sebelum disimpan
# rekap[tanggal_hari_ini] = rekap[tanggal_hari_ini].fillna("")
# rekap = rekap.sort_values(
#     ["level_6_code", "assignment_id", "source_type", "anomali_no"]
# ).reset_index(drop=True)

# 5. Simpan
nama_file_baru = FOLDER_REKAP / f"rekap_anomali_{tanggal_jam}.xlsx"
simpan_ke_excel(rekap, ringkasan, nama_file_baru)
print("Disimpan ke:", nama_file_baru)

Anomali resolved: 30716 | belum resolved: 125067
File rekap terakhir ditemukan: ..\rekap_anomali_pertanggal\rekap_anomali_14-07-2026_10-52-51.xlsx
Kolom tanggal terdeteksi: ['06-07-2026', '07-07-2026', '08-07-2026', '09-07-2026', '10-07-2026', '11-07-2026', '13-07-2026', '14-07-2026', '15-07-2026']
                                                             Keterangan  Jumlah
                                                        Data sebelumnya  139526
                                                        Data baru masuk  125067
Perubahan (centang -> blank/Kondisi Lapangan), 14-07-2026 -> 15-07-2026    1340
                Belum berubah (tetap centang), 14-07-2026 -> 15-07-2026  123897
                                                 Total setelah digabung  139640
Disimpan ke: ..\rekap_anomali_pertanggal\rekap_anomali_15-07-2026_09-55-50.xlsx
